### Resumen - Gasto por ubicación
> Para cada "País, Ciudad y Departamento" se necesita saber los siguientes gastos:
1. Cantidad de empleados
2. Total de gasto
3. Promedio de gasto

In [0]:
df_locations = spark.table("zenviro.silver.locations_py").alias("l")
display(df_locations)

In [0]:
df_departments = spark.table("zenviro.silver.departments_py").alias("d")
display(df_departments)

In [0]:
df_employees = spark.table("zenviro.silver.employees_py").alias("e")
display(df_employees)

In [0]:
df_employment_histories = spark.table("zenviro.silver.employment_histories_py").alias("eh")
display(df_employment_histories)

In [0]:
from pyspark.sql.functions import col, count, sum, avg, round

df_employees_by_location_summary = (
    df_locations
    .join(df_departments, col("l.location_id") == col("d.location_id"), "inner")
    .join(df_employees, col("d.department_id") == col("e.department_id"), "inner")
    .join(df_employment_histories, col("e.employee_id") == col("eh.employee_id"), "inner")
    .groupBy("l.country", "l.city", "d.department_name")
    .agg(
        count("e.employee_id").alias("total_employees"),
        sum("eh.salary").alias("total_salary"),
        round(avg("eh.salary"), 2).alias("avg_salary")
    )
    .orderBy("l.country", "l.city", "d.department_name")
)
display(df_employees_by_location_summary)

In [0]:
df_employees_by_location_summary.writeTo("zenviro.gold.employees_by_location_summary_py").createOrReplace()

In [0]:
display(spark.table("zenviro.gold.employees_by_location_summary_py"))